# Predict — BasicTS benchmark on Colab Pro (Person A, Day 1–3)

Runs the official BasicTS harness (TKDE 2024) with the **official model configs** for STID, Graph WaveNet, DCRNN, AGCRN and STAEformer on METR-LA / PEMS-BAY / PEMS08, then a Chronos-Bolt zero-shot comparison. Every checkpoint and test log is written to Drive.

Runtime → GPU (A100 or L4). Expect: STID ≈ 10 min, GWNet ≈ 1–2 h, STAEformer ≈ 2–3 h per dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os, pathlib
DRIVE = pathlib.Path('/content/drive/MyDrive/dhakaflow/predict'); DRIVE.mkdir(parents=True, exist_ok=True)
!nvidia-smi -L

## 1. Harness at a pinned commit (never modify upstream code)

In [ ]:
BASICTS_SHA = 'c2bb6e31'   # pin recorded in upstream/UPSTREAM_PINS.yaml — re-verify if training fails
%cd /content
!git clone -q https://github.com/GestaltCogTeam/BasicTS.git
%cd /content/BasicTS
!git checkout -q $BASICTS_SHA && git log --oneline -1
!pip install -q -r requirements.txt
!pip install -q -e .

## 2. Raw data
BasicTS ships pre-processed datasets via the links in its README (`datasets/README.md`: Google Drive / Baidu). Download once, keep them on Drive. If `gdown` fails, download the folder manually into `MyDrive/dhakaflow/predict/raw_data/` and re-run.

In [ ]:
!pip install -q gdown
RAW = DRIVE/'raw_data'
if not (RAW/'METR-LA').exists():
    print('>>> Follow datasets/README.md in BasicTS and place METR-LA, PEMS-BAY, PEMS08 under', RAW)
    !cat datasets/README.md | head -40
else:
    !rm -rf datasets/raw_data && ln -s {RAW} datasets/raw_data
    for ds in ['METR-LA','PEMS-BAY','PEMS08']:
        !python scripts/data_preparation/{ds}/generate_training_data.py

## 3. Train — official configs, one command each (queue them; Colab keeps running while you sleep)

In [ ]:
import subprocess, itertools, time
MODELS   = ['STID', 'GWNet', 'AGCRN', 'DCRNN', 'STAEformer']   # order = cheapest first
DATASETS = ['METR-LA', 'PEMS-BAY', 'PEMS08']
CKPT = DRIVE/'ckpt'; CKPT.mkdir(exist_ok=True)
for m, d in itertools.product(MODELS, DATASETS):
    cfg = f'baselines/{m}/{d}.py'
    if not os.path.exists(cfg):
        print('no official config for', m, d, '- skipped'); continue
    log = DRIVE/f'logs/{m}_{d}.log'; log.parent.mkdir(exist_ok=True)
    if log.exists() and 'test' in log.read_text()[-4000:].lower():
        print('done already:', m, d); continue
    t0 = time.time()
    print('=== training', m, d, '->', log)
    with open(log, 'w') as f:
        subprocess.run(['python', 'experiments/train.py', '-c', cfg, '-g', '0'], stdout=f, stderr=subprocess.STDOUT)
    !cp -r checkpoints/{m} {CKPT}/ 2>/dev/null || true
    print(f'    {(time.time()-t0)/60:.1f} min')

## 4. Collect test MAE / RMSE / MAPE at 15 / 30 / 60 min into one CSV (next to the published numbers)

In [ ]:
import re, csv, glob
PUBLISHED = {('GWNet','METR-LA'):(2.69,3.07,3.53), ('GWNet','PEMS-BAY'):(1.30,1.63,1.95),
             ('DCRNN','METR-LA'):(2.77,3.15,3.60), ('DCRNN','PEMS-BAY'):(1.38,1.74,2.07)}
rows=[]
for log in sorted(glob.glob(str(DRIVE/'logs/*.log'))):
    m, d = pathlib.Path(log).stem.split('_',1)
    txt = open(log).read()
    # BasicTS prints e.g. "Evaluate best model on test data for horizon 3, Test MAE: 2.69, Test RMSE: 5.15, Test MAPE: 0.0688"
    for h, mae, rmse, mape in re.findall(r'horizon\s+(\d+).*?MAE:\s*([\d.]+).*?RMSE:\s*([\d.]+).*?MAPE:\s*([\d.]+)', txt, re.S):
        if int(h) in (3,6,12):
            pub = PUBLISHED.get((m,d)); k={3:0,6:1,12:2}[int(h)]
            rows.append(dict(model=m, dataset=d, horizon_min=int(h)*5, mae=float(mae), rmse=float(rmse), mape=float(mape),
                             published_mae=pub[k] if pub else ''))
out = DRIVE/'results'; out.mkdir(exist_ok=True)
with open(out/'benchmark.csv','w',newline='') as f:
    w=csv.DictWriter(f, fieldnames=list(rows[0]) if rows else ['model']); w.writeheader(); w.writerows(rows)
import pandas as pd; pd.DataFrame(rows)

## 5. Novelty row — Chronos-Bolt zero-shot on the same test windows (no training)

In [ ]:
!pip install -q chronos-forecasting
import numpy as np, torch
from chronos import BaseChronosPipeline
pipe = BaseChronosPipeline.from_pretrained('amazon/chronos-bolt-base', device_map='cuda', torch_dtype=torch.bfloat16)
for d in DATASETS:
    # BasicTS stores the normalised tensor + index; use its own dataset class to get identical test windows:
    from basicts.data import TimeSeriesForecastingDataset
    ds = TimeSeriesForecastingDataset(dataset_name=d, train_val_test_ratio=[0.7,0.1,0.2] if d!='PEMS08' else [0.6,0.2,0.2], input_len=12, output_len=12, mode='test')
    maes = {3:[],6:[],12:[]}
    for i in range(0, len(ds), 50):                      # subsample windows for speed; report n
        x, y = ds[i]['inputs'][...,0], ds[i]['target'][...,0]     # (12, N), (12, N)
        ctx = torch.tensor(x.T)                          # N series × 12 steps
        q, mean = pipe.predict_quantiles(context=ctx, prediction_length=12, quantile_levels=[0.5])
        pred = mean.numpy().T                            # (12, N)
        for h in maes: maes[h].append(np.abs(pred[h-1]-y[h-1]).mean())
    print(d, {f'{h*5}min': round(float(np.mean(v)),3) for h,v in maes.items()}, 'windows', len(maes[3]))

**Note on the Chronos cell:** BasicTS z-scores inputs; Chronos must see raw speeds. Check `ds.scaler`/`datasets/<d>/scaler.pkl` and de-normalise `x, y` before calling `predict_quantiles`, or the MAE row is meaningless. Record the window count you used.